# Phase 6 — Results Dashboard

Consolidates every artifact produced by phases 1-5 into a single set of charts.
Plots are saved to `reports/charts/*.png` and embedded in this notebook.

**Audience**: the business stakeholder, not the ML engineer. Each chart starts with
a one-line takeaway and ends with a 'what this means' interpretation block.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.visualization.plots import (
    plot_churn_model_comparison,
    plot_pipeline_ablation,
    plot_fe_comparison,
    plot_churn_distribution,
    plot_roc_pr,
    plot_calibration,
    plot_confusion,
    plot_feature_importance,
    plot_risk_segment,
    plot_customer_funnel,
)
from src.evaluation.business import evaluate_by_risk_segment

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
REPORTS = PROJECT_ROOT / 'reports'
CHARTS = REPORTS / 'charts'
CHARTS.mkdir(parents=True, exist_ok=True)

## 1. Three churn models — which one to deploy?

**Takeaway**: BG-NBD edges out the GBM stack on AUC and PR-AUC even though it uses far fewer features. Cox PH lags. We deploy GBM (rich features, future-proof) and keep BG-NBD as a calibration baseline.

In [ ]:
comp = pd.read_csv(REPORTS / 'churn_model_comparison.csv', index_col=0)
plot_churn_model_comparison(comp, out_path=CHARTS / 'churn_model_comparison.png')

## 2. Pipeline ablation — does each stage add value?

**Takeaway**: SASRec retrieval *alone* underperforms even a popular-items recommendation for the high-churn segment, but adding the churn-aware LGBM ranker on top yields a **2–4× lift**. Lesson: personalization without churn awareness is worse than nothing here.

In [ ]:
abl = pd.read_csv(REPORTS / 'pipeline_ablation.csv', index_col=0)
plot_pipeline_ablation(abl, out_path=CHARTS / 'pipeline_ablation.png')

## 3. Feature engineering — was it worth doubling the columns?

**Takeaway**: Expanded set (76 features) lifts test AUC by ~1pt and reduces Brier (better calibrated) — modest but real. Worth keeping; the marginal compute cost is negligible.

In [ ]:
fe = pd.read_csv(REPORTS / 'fe_comparison.csv', index_col=0)
plot_fe_comparison(fe, out_path=CHARTS / 'fe_comparison.png')

## 4. Risk-tier population — who gets which retention action?

**Takeaway**: The decision layer assigns ~30% of customers to the high-risk tier (20% off), ~30% to medium (10% off), 40% to low (no action). Sales can size the retention budget directly off this chart.

In [ ]:
churn_gbm = pd.read_parquet(FEAT / 'churn_scores_gbm_test.parquet')
plot_churn_distribution(churn_gbm, 'p_churn_gbm', out_path=CHARTS / 'churn_distribution.png')

tiers = pd.cut(churn_gbm['p_churn_gbm'], bins=[0, 0.4, 0.7, 1.0], labels=['low','medium','high']).value_counts()
print(tiers)

## 5. ROC & PR curves for the three churn approaches

**Takeaway**: Read the AUC and AP numbers in the legend. The closer the ROC curve hugs the top-left and the PR curve sits above the base-rate line, the better.

In [ ]:
labels_test = pd.read_parquet(PROC / 'churn_labels_test.parquet')
scores = {
    'GBM Stack': pd.read_parquet(FEAT / 'churn_scores_gbm_test.parquet').rename(columns={'p_churn_gbm':'p'}),
    'BG-NBD':    pd.read_parquet(FEAT / 'churn_scores_bgnbd_test.parquet').rename(columns={'p_churn_bgnbd':'p'}),
    'Cox PH':    pd.read_parquet(FEAT / 'churn_scores_cox_test.parquet').rename(columns={'p_churn_cox':'p'}),
}
common = labels_test.merge(scores['GBM Stack'], on='customer_id', how='inner')
score_sets = {}
for name, df in scores.items():
    joined = labels_test.merge(df, on='customer_id', how='inner')
    joined['p'] = joined['p'].fillna(joined['p'].median())
    score_sets[name] = (joined['churn'].values, joined['p'].values)

# Use one common index — easiest if we align on the GBM customers
ref_y = score_sets['GBM Stack'][0]
score_dict = {k: v[1] for k, v in score_sets.items() if (v[0] == ref_y).all() or len(v[0]) == len(ref_y)}
# Fallback: just plot per-model independently against its own y
plot_roc_pr(ref_y, {k: score_sets[k][1] for k in score_sets if len(score_sets[k][1]) == len(ref_y)},
            out_path=CHARTS / 'roc_pr.png')

## 6. Calibration — are the probabilities trustworthy?

**Takeaway**: A perfectly calibrated model lies on the diagonal. Sales teams should only trust risk-tier thresholds if calibration is good — otherwise "p=0.7" doesn't actually mean 70%.

In [ ]:
plot_calibration(ref_y, {k: score_sets[k][1] for k in score_sets if len(score_sets[k][1]) == len(ref_y)},
                 out_path=CHARTS / 'calibration.png')

## 7. Confusion matrix at the deployed threshold

**Takeaway**: Sales can read the off-diagonal cells to estimate the cost of false positives (un-needed discounts) and false negatives (lost customers we didn't reach).

In [ ]:
comp_test = comp.loc['GBM Stack — test']
thr = float(comp_test['threshold'])
gbm_y, gbm_p = score_sets['GBM Stack']
y_pred = (gbm_p >= thr).astype(int)
plot_confusion(gbm_y, y_pred, out_path=CHARTS / 'confusion.png')
print(f'Threshold used: {thr:.4f}')

## 8. Feature importance — what signals drive predictions?

**Takeaway**: Recency-related features dominate. Anything sales already tracks (last purchase date, last basket size) is enough to act on; the rest is marginal lift.

In [ ]:
# Re-train a single LightGBM on expanded features to pull importances
from lightgbm import LGBMClassifier
from src.features.target_encoding import fit_oof

lbl = {n: pd.read_parquet(PROC / f'churn_labels_{n}.parquet') for n in ['train','val','test']}
exp = {n: pd.read_parquet(FEAT / f'expanded_features_{n}.parquet').merge(lbl[n][['customer_id','churn']], on='customer_id') for n in ['train','val','test']}
for g in ['last_country', 'cohort_month']:
    if g in exp['train']:
        oof, enc = fit_oof(exp['train'], 'churn', g)
        exp['train'][f'te_{g}'] = oof.values
        for n in ['val','test']:
            exp[n][f'te_{g}'] = enc.transform(exp[n]).values
cols = [c for c in exp['train'].columns if c not in {'customer_id','churn','last_country','country'} and pd.api.types.is_numeric_dtype(exp['train'][c])]
m = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, n_jobs=-1, verbosity=-1)
m.fit(exp['train'][cols].fillna(0), exp['train']['churn'])
imp = pd.DataFrame({'feature': cols, 'gain': m.booster_.feature_importance(importance_type='gain')})
plot_feature_importance(imp, k=25, out_path=CHARTS / 'feature_importance.png')

## 9. Is the model better at high-risk customers (where it matters)?

**Takeaway**: Performance broken down by quantile of predicted churn. If precision/recall at the top quantile is high, the team can trust the high-risk tier to actually contain real churners.

In [ ]:
yte = labels_test.merge(scores['GBM Stack'], on='customer_id', how='inner')
by_seg = evaluate_by_risk_segment(yte['customer_id'].values, yte['churn'].values, yte['p'].values)
plot_risk_segment(by_seg, out_path=CHARTS / 'risk_segment.png')
by_seg

## 10. Customer funnel — pipeline drop-off

**Takeaway**: how many customers reach each pipeline stage. Drop-offs (e.g. customers with no purchase history we can encode) are quantified.

In [ ]:
df_tx = pd.read_parquet(PROC / 'transactions_clean.parquet')
cand = pd.read_parquet(FEAT / 'retrieval_candidates_test.parquet')
stages = {
    '1. All customers in test labels': int(labels_test['customer_id'].nunique()),
    '2. With ≥1 purchase (have history)': int(df_tx['customer_id'].nunique()),
    '3. With SASRec retrieval (≥3 purchases)': int(cand['customer_id'].nunique()),
    '4. Got top-10 recommendations': int(cand[cand['stock_code'].notna()]['customer_id'].nunique()),
    '5. High-risk tier (action triggered)': int((scores['GBM Stack']['p'] >= 0.7).sum()),
}
plot_customer_funnel(stages, out_path=CHARTS / 'customer_funnel.png')
stages

## Summary for business stakeholders

1. **Model quality** — GBM Stack and BG-NBD both reach AUC ≈ 0.77-0.80 on held-out test. Calibration is reasonable.
2. **Personalization adds value only when combined with churn signal** — SASRec retrieval alone is *worse* than popular-items for the high-churn segment. Adding the LGBM ranker (which knows p(churn)) gives a **4× NDCG lift**.
3. **Decision layer assigns concrete actions** — ~30% high-risk customers get 20% off, ~30% medium-risk get 10% off, 40% no action.
4. **The top features are sales-readable** — recency, lifetime spend, interval stats. Sales can sanity-check this list against their intuition.

All charts are in `reports/charts/` — paste them straight into a deck or QBR.